# Silver One-Time Fixes

Run **once, manually**, after importing the new silver notebooks. **Not part of the scheduled job.**

Silver currently holds rows that the (new) rules would reject:
- customers with invalid gender or impossible dates (65 + 512 at the last check)
- claims validated against **bronze** policies, so some may point to policies silver rejected
- agents validated against **bronze** branches

For each bad silver row this notebook **resets its bronze rows to `merge_flag = false` first, then deletes it from silver**,
and cascades down (customer -> policy -> claim, branch -> agent). The next normal job run then re-processes those bronze rows:
bad data goes to quarantine, children of missing parents wait as soft rejects.

**How to use**
1. Run all with `confirm = NO` -> preview only, nothing changes.
2. Decide on the customer rules (see the preview). If you want to keep those customers instead, set
   `include_customer_cleanup = NO` and relax the rule in Silver_Customer.
3. Set `confirm = YES` and run all again.

Every step is safe to re-run: once the bad rows are gone, each statement affects 0 rows.

In [0]:
dbutils.widgets.dropdown("confirm", "NO", ["NO", "YES"], "Apply changes")
dbutils.widgets.dropdown("include_customer_cleanup", "YES", ["YES", "NO"], "Include customer rule cleanup")

S = "policysystemdatabricks.silver"
B = "policysystemdatabricks.bronze"

BAD_CUSTOMER = """(gender IS NULL OR gender NOT IN ('Male', 'Female')
       OR registration_date IS NULL OR date_of_birth IS NULL
       OR registration_date <= date_of_birth)"""

include_customers = dbutils.widgets.get("include_customer_cleanup") == "YES"

In [0]:
%sql
WITH bad_customers AS (
  SELECT customer_id FROM policysystemdatabricks.silver.customer
  WHERE gender IS NULL OR gender NOT IN ('Male', 'Female')
     OR registration_date IS NULL OR date_of_birth IS NULL
     OR registration_date <= date_of_birth
),
policies_of_bad_customers AS (
  SELECT p.policy_id FROM policysystemdatabricks.silver.policydata p JOIN bad_customers c USING (customer_id)
)
SELECT
  (SELECT count(*) FROM bad_customers)                                   AS customers_to_remove,
  (SELECT count(*) FROM policysystemdatabricks.silver.customer)          AS customers_total,
  (SELECT count(*) FROM policies_of_bad_customers)                       AS policies_that_would_lose_customer,
  (SELECT count(*) FROM policysystemdatabricks.silver.claim c
    WHERE c.policy_id IN (SELECT policy_id FROM policies_of_bad_customers)) AS claims_that_would_lose_policy,
  (SELECT count(*) FROM policysystemdatabricks.silver.claim c
    LEFT ANTI JOIN policysystemdatabricks.silver.policydata p ON c.policy_id = p.policy_id) AS claims_already_without_silver_policy,
  (SELECT count(*) FROM policysystemdatabricks.silver.agent a
    LEFT ANTI JOIN policysystemdatabricks.silver.branch b ON a.branch_id = b.branch_id)     AS agents_without_silver_branch

customers_to_remove,customers_total,policies_that_would_lose_customer,claims_that_would_lose_policy,claims_already_without_silver_policy,agents_without_silver_branch
512,968,416,402,218,0


In [0]:
if dbutils.widgets.get("confirm") != "YES":
    dbutils.notebook.exit("Preview only. Set confirm = YES and run all again to apply the fixes.")

In [0]:
def run(label, sql):
    affected = spark.sql(sql).first()["num_affected_rows"]
    print(f"{label:55s} {affected}")


# 1. Customers that fail the gender / date rules
if include_customers:
    run("bronze.customer reset to merge_flag = false", f"""
        UPDATE {B}.customer SET merge_flag = false
        WHERE customer_id IN (SELECT customer_id FROM {S}.customer WHERE {BAD_CUSTOMER})""")
    run("silver.customer deleted", f"DELETE FROM {S}.customer WHERE {BAD_CUSTOMER}")

# 2. Policies whose customer is no longer in silver
run("bronze.policydata reset to merge_flag = false", f"""
    UPDATE {B}.policydata SET merge_flag = false
    WHERE policy_id IN (SELECT p.policy_id FROM {S}.policydata p
                        LEFT ANTI JOIN {S}.customer c ON p.customer_id = c.customer_id)""")
run("silver.policydata deleted", f"""
    DELETE FROM {S}.policydata AS t
    WHERE NOT EXISTS (SELECT 1 FROM {S}.customer c WHERE c.customer_id = t.customer_id)""")

# 3. Claims whose policy is not in silver (covers both old bronze-validated claims and the cascade above)
run("bronze.claim reset to merge_flag = false", f"""
    UPDATE {B}.claim SET merge_flag = false
    WHERE claim_id IN (SELECT c.claim_id FROM {S}.claim c
                       LEFT ANTI JOIN {S}.policydata p ON c.policy_id = p.policy_id)""")
run("silver.claim deleted", f"""
    DELETE FROM {S}.claim AS t
    WHERE NOT EXISTS (SELECT 1 FROM {S}.policydata p WHERE p.policy_id = t.policy_id)""")

# 4. Agents whose branch is not in silver
run("bronze.agent reset to merge_flag = false", f"""
    UPDATE {B}.agent SET merge_flag = false
    WHERE agent_id IN (SELECT a.agent_id FROM {S}.agent a
                       LEFT ANTI JOIN {S}.branch b ON a.branch_id = b.branch_id)""")
run("silver.agent deleted", f"""
    DELETE FROM {S}.agent AS t
    WHERE NOT EXISTS (SELECT 1 FROM {S}.branch b WHERE b.branch_id = t.branch_id)""")

print("\nDone. Now run the silver job once: rejected rows go to quarantine, orphans wait as soft rejects.")

bronze.customer reset to merge_flag = false             1024
silver.customer deleted                                 512
bronze.policydata reset to merge_flag = false           1248
silver.policydata deleted                               416
bronze.claim reset to merge_flag = false                620
silver.claim deleted                                    620
bronze.agent reset to merge_flag = false                0
silver.agent deleted                                    0

Done. Now run the silver job once: rejected rows go to quarantine, orphans wait as soft rejects.
